In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import pytorch_lightning as pl


from domains.previsao_ceu.refactored_module import build_preprocessing_pipeline

import yaml

def load_config():
    with open('configs/ceu_config.yaml', 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)

CONFIG_CEU = load_config()

In [2]:
# 2. Leitura
csv_path = CONFIG_CEU['csv_path']
print(f"⏳ Lendo: {csv_path}")
df = pd.read_csv(csv_path)

if 'Date_Time' in df.columns:
    df['Date_Time'] = pd.to_datetime(df['Date_Time'])
    df = df.drop_duplicates(subset=['Date_Time'], keep='first').set_index('Date_Time').sort_index()
df = df[~df.index.duplicated(keep='first')]

df['Temperatura ambiente °C'].loc[df['Temperatura ambiente °C'] < 0] = np.nan
df['Umidade Relativa %'].loc[df['Umidade Relativa %'] < 0] = np.nan

# 3. Pré-processamento
pp_conf = CONFIG_CEU['preprocessing']

# Instancia passando o mapa explícito. 
# Isso garante que a padronização aconteça conforme o CONFIG_CEU acima.
preprocessor = build_preprocessing_pipeline(
    latitude=pp_conf['latitude'], 
    longitude=pp_conf['longitude'], 
    altitude=pp_conf['altitude'],
    timezone=pp_conf['timezone'], 
    nominal_power=pp_conf['nominal_power'], 
    start_year=pp_conf['start_year'],
    features_to_scale=pp_conf['features_to_scale'],
    target_col=CONFIG_CEU['prediction_mode'], # <--- unica variavel que não vem do preprocessing
    column_mapping=pp_conf['column_mapping'],
    cs_model = 'esra',
    kasten_corr=True
)

preprocessor.fit(df)

# O método transform usa o column_mapping para renomear as colunas
df_processed = preprocessor.transform(df)

⏳ Lendo: data/pv0.csv
Otimizando TL para 381 dias selecionados...
Otimização concluída. Média TL: 6.28


In [9]:
from skills_codelab.refactored_pipeline.pipeline import build_solar_pipeline

new_pipeline = build_solar_pipeline(latitude=pp_conf['latitude'], 
                                longitude=pp_conf['longitude'], 
                                altitude=pp_conf['altitude'],
                                timezone=pp_conf['timezone'], 
                                nominal_power=pp_conf['nominal_power'], 
                                start_year=pp_conf['start_year'],
                                features_to_scale=pp_conf['features_to_scale'],
                                target_col=CONFIG_CEU['prediction_mode'], # <--- unica variavel que não vem do preprocessing
                                column_mapping=pp_conf['column_mapping'],
                                cs_model = 'esra',
                                kasten_corr=True)

In [ ]:
df_new_result = new_pipeline.fit_transform(df)


Otimizando TL para 381 dias selecionados...
Otimização concluída. Média TL: 6.28


,target,ghi,irrad_poa,temp_amb,humidity,wind_speed,Pressão Baromêtrica mm Hg,rain,zenith,apparent_zenith,...,qc_reprovals,P_fracao_difusa,P_kt,P1,P2,P3,irr_clearsky_ratio,sin_elevation,delta_kt,delta_fracao_difusa
Date_Time,,,,,,,,,,,,,,,,,,,,,
2015-06-11 18:00:00-03:00,0.000000,0.0,0.0,0.298206,0.6308,0.041431,616.80,0.0,97.626451,97.626451,...,alpha;ghi_limits;dhi_limits;closure;ghi_min_sl...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.132714,0.000000,0.00000
2015-06-11 19:00:00-03:00,0.000000,0.0,0.0,0.484224,0.6113,0.001412,595.90,0.0,110.587917,110.587917,...,alpha;ghi_limits;dhi_limits;closure;ghi_min_sl...,0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,-0.351644,0.000000,0.00000
2015-06-11 20:00:00-03:00,0.000000,0.0,0.0,0.612704,0.5979,0.000000,576.90,0.0,123.868873,123.868873,...,alpha;ghi_limits;dhi_limits;closure;ghi_min_sl...,0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,-0.557294,0.000000,0.00000
2015-06-11 21:00:00-03:00,0.000000,0.0,0.0,0.640132,0.5949,0.000000,563.70,0.0,137.362677,137.362677,...,alpha;ghi_limits;dhi_limits;closure;ghi_min_sl...,0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,-0.735656,0.000000,0.00000
2015-06-11 22:00:00-03:00,0.000000,0.0,0.0,0.633326,0.5950,0.000000,554.70,0.0,150.995594,150.995594,...,alpha;ghi_limits;dhi_limits;closure;ghi_min_sl...,0.000000,-0.000000,0.000000,0.000000,0.000000,0.000000,-0.874582,0.000000,0.00000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2022-12-31 19:00:00-03:00,0.000371,29.0,25.0,0.450815,0.9220,0.000000,700.04,0.0,91.482672,91.482672,...,alpha;ghi_limits;dhi_limits;closure;dni_limits...,0.933366,0.274216,0.003821,0.005831,0.002067,0.862069,-0.025875,-0.274216,-0.10578
2022-12-31 20:00:00-03:00,0.000000,0.0,0.0,0.443803,0.8840,0.000942,700.66,0.0,103.460544,103.460544,...,alpha;ghi_limits;dhi_limits;closure;ghi_min_sl...,0.827586,0.000000,0.000401,0.003821,0.005831,0.000000,-0.232776,-0.000000,0.00000
2022-12-31 21:00:00-03:00,0.000000,0.0,0.0,0.423592,0.9260,0.000471,700.71,0.0,114.404647,114.404647,...,alpha;ghi_limits;dhi_limits;closure;ghi_min_sl...,0.000000,-0.000000,0.000000,0.000401,0.003821,0.000000,-0.413178,0.000000,0.00000


In [11]:
# 5. Validação Científica
print("\n--- RESULTADOS DA COMPARAÇÃO ---")

# 5.1. Verifica se as colunas geradas são exatamente as mesmas
colunas_antigas = set(df_processed.columns)
colunas_novas = set(df_new_result.columns)

if colunas_antigas != colunas_novas:
    print("❌ Diferença nas colunas encontradas!")
    print(f"Faltando no novo: {colunas_antigas - colunas_novas}")
    print(f"A mais no novo: {colunas_novas - colunas_antigas}")
else:
    print("✅ Todas as colunas batem perfeitamente.")

# 5.2. Verifica se os valores matemáticos são idênticos (com tolerância de 1e-5 para arredondamentos)
try:
    pd.testing.assert_frame_equal(
        df_processed, 
        df_new_result, 
        check_exact=False, 
        rtol=1e-5, 
        atol=1e-5
    )
    print("✅ SUCESSO ABSOLUTO: Os DataFrames são matematicamente idênticos!")
    print("A refatoração SOLID foi um sucesso. O 'God Object' pode ser aposentado.")
except AssertionError as e:
    print("❌ DIFERENÇA DE VALORES ENCONTRADA:")
    print("O erro do Pandas apontou a seguinte discrepância nas matrizes:")
    print(e)


--- RESULTADOS DA COMPARAÇÃO ---
✅ Todas as colunas batem perfeitamente.
❌ DIFERENÇA DE VALORES ENCONTRADA:
O erro do Pandas apontou a seguinte discrepância nas matrizes:
DataFrame.iloc[:, 3] (column name="temp_amb") are different

DataFrame.iloc[:, 3] (column name="temp_amb") values are different (99.62109 %)
[index]: [2015-06-11 18:00:00-03:00, 2015-06-11 19:00:00-03:00, 2015-06-11 20:00:00-03:00, 2015-06-11 21:00:00-03:00, 2015-06-11 22:00:00-03:00, 2015-06-11 23:00:00-03:00, 2015-06-12 00:00:00-03:00, 2015-06-12 01:00:00-03:00, 2015-06-12 02:00:00-03:00, 2015-06-12 03:00:00-03:00, 2015-06-12 04:00:00-03:00, 2015-06-12 05:00:00-03:00, 2015-06-12 06:00:00-03:00, 2015-06-12 07:00:00-03:00, 2015-06-12 08:00:00-03:00, 2015-06-12 09:00:00-03:00, 2015-06-12 10:00:00-03:00, 2015-06-12 11:00:00-03:00, 2015-06-12 12:00:00-03:00, 2015-06-12 13:00:00-03:00, 2015-06-12 14:00:00-03:00, 2015-06-12 15:00:00-03:00, 2015-06-12 16:00:00-03:00, 2015-06-12 17:00:00-03:00, 2015-06-12 18:00:00-03:00, 20

In [13]:
df_processed.iloc[:, 3]

Date_Time
2015-06-11 18:00:00-03:00    0.243879
2015-06-11 19:00:00-03:00    0.444296
2015-06-11 20:00:00-03:00    0.582722
2015-06-11 21:00:00-03:00    0.612274
2015-06-11 22:00:00-03:00    0.604942
                               ...   
2022-12-31 19:00:00-03:00    0.408301
2022-12-31 20:00:00-03:00    0.400747
2022-12-31 21:00:00-03:00    0.378972
2022-12-31 22:00:00-03:00    0.363196
2022-12-31 23:00:00-03:00    0.365418
Name: temp_amb, Length: 66242, dtype: float64